<a href="https://colab.research.google.com/github/Fr4110/Human-Motion/blob/main/HumanMotionComputing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip uninstall -y tensorflow tensorflow-decision-forests ydf grain tf-keras -q
!pip install -q "protobuf>=4.25.3,<5"
!pip install -q mediapipe==0.10.14
!pip install -q ultralytics opencv-python-headless

import cv2, numpy as np, time, os, json
import matplotlib.pyplot as plt
%matplotlib inline

os.makedirs("/content/videos", exist_ok=True)
os.makedirs("/content/results", exist_ok=True)


In [ ]:
import cv2, numpy as np, time, os, json
import matplotlib.pyplot as plt
%matplotlib inline

import mediapipe as mp
mp_pose = mp.solutions.pose


In [ ]:
from google.colab import files
import shutil

VIDEO_DIR = "/content/videos"
RESULTS_DIR = "/content/results"
os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Seleziona i 3 file video (tai-chi 1.mp4, tai-chi 2.mp4, tai-chi 3.mp4)")
uploaded = files.upload()

for fname in uploaded.keys():
    shutil.move(fname, f"{VIDEO_DIR}/{fname}")

video_files = sorted(os.listdir(VIDEO_DIR))
print("\nVideo caricati:")
for v in video_files:
    path = f"{VIDEO_DIR}/{v}"
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    print(f" - {v}: {w}x{h}, {fps:.1f} fps, {n_frames} frame, {n_frames/fps:.1f} sec")


In [ ]:
video_files = sorted(os.listdir(VIDEO_DIR))
print("Video disponibili:", video_files)

def get_video_frames(video_path, max_frames=None, resize=None):
    """Legge un video e restituisce una lista di frame (BGR, formato OpenCV)."""
    cap = cv2.VideoCapture(video_path)
    frames = []
    count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if resize is not None:
            frame = cv2.resize(frame, resize)
        frames.append(frame)
        count += 1
        if max_frames is not None and count >= max_frames:
            break
    cap.release()
    return frames

def run_pose_on_video(video_path, pose_fn, max_frames=None):
    """Esegue pose_fn su tutti i frame di un video, misurando tempo e
    rilevamento, e restituisce summary, tempi e conteggi keypoint."""
    frames = get_video_frames(video_path, max_frames=max_frames)
    n_frames = len(frames)

    times = []
    keypoints_per_frame = []
    failed_frames = 0

    for frame in frames:
        t0 = time.time()
        result = pose_fn(frame)
        t1 = time.time()
        times.append(t1 - t0)

        if result is None or result[0] is None:
            failed_frames += 1
            keypoints_per_frame.append(0)
        else:
            keypoints_per_frame.append(result[1])

    summary = {
        "video": os.path.basename(video_path),
        "n_frames": n_frames,
        "avg_time_per_frame": float(np.mean(times)),
        "fps_processing": 1.0 / np.mean(times) if np.mean(times) > 0 else 0,
        "total_time": float(np.sum(times)),
        "avg_keypoints_detected": float(np.mean(keypoints_per_frame)),
        "failed_frames": failed_frames,
        "failed_frames_pct": 100.0 * failed_frames / n_frames if n_frames > 0 else 0,
    }
    return summary, times, keypoints_per_frame

print("Funzioni comuni definite.")


In [ ]:
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

pose_mediapipe = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def mediapipe_pose_fn(frame_bgr):
    """Restituisce (keypoints_array, n_keypoints_rilevati) o (None, 0)"""
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    results = pose_mediapipe.process(frame_rgb)

    if results.pose_landmarks is None:
        return None, 0

    landmarks = results.pose_landmarks.landmark
    keypoints = np.array([[lm.x, lm.y, lm.visibility] for lm in landmarks])
    n_detected = int(np.sum(keypoints[:, 2] > 0.5))
    return keypoints, n_detected

test_video = f"{VIDEO_DIR}/{video_files[0]}"
summary, times, kp_counts = run_pose_on_video(test_video, mediapipe_pose_fn, max_frames=60)
print("Test MediaPipe (60 frame):")
for k, v in summary.items():
    print(f"  {k}: {v}")


In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolov8n-pose.pt")

def yolo_pose_fn(frame_bgr):
    """Restituisce (keypoints_array, n_keypoints_rilevati) o (None, 0)"""
    results = yolo_model(frame_bgr, verbose=False)

    if len(results) == 0 or results[0].keypoints is None or len(results[0].keypoints.xy) == 0:
        return None, 0

    keypoints_xy = results[0].keypoints.xy[0].cpu().numpy()
    keypoints_conf = results[0].keypoints.conf[0].cpu().numpy() if results[0].keypoints.conf is not None else None

    if keypoints_conf is not None:
        n_detected = int(np.sum(keypoints_conf > 0.5))
    else:
        n_detected = int(np.sum(np.any(keypoints_xy > 0, axis=1)))

    return keypoints_xy, n_detected

summary_yolo, times_yolo, kp_counts_yolo = run_pose_on_video(test_video, yolo_pose_fn, max_frames=60)
print("Test YOLO Pose (60 frame):")
for k, v in summary_yolo.items():
    print(f"  {k}: {v}")


In [ ]:
!apt-get update -qq
!apt-get install -y -qq cmake libgflags-dev libgoogle-glog-dev \
    libprotobuf-dev protobuf-compiler libgoogle-perftools-dev \
    libboost-all-dev libhdf5-dev libatlas-base-dev libopencv-dev

In [ ]:
%cd /content
!git clone -q --depth 1 https://github.com/CMU-Perceptual-Computing-Lab/openpose.git
%cd openpose
!git submodule update -q --init --recursive --remote

In [ ]:
!pip install -q kagglehub
import kagglehub
import shutil

kaggle_path = kagglehub.dataset_download("changethetuneman/openpose-model")
print("Modelli scaricati in:", kaggle_path)

OPENPOSE_MODELS = "/content/openpose/models"
mapping = {
    "pose_iter_584000.caffemodel": f"{OPENPOSE_MODELS}/pose/body_25/",
    "pose_iter_440000.caffemodel": f"{OPENPOSE_MODELS}/pose/coco/",
    "pose_iter_160000.caffemodel": f"{OPENPOSE_MODELS}/pose/mpi/",
    "pose_iter_116000.caffemodel": f"{OPENPOSE_MODELS}/face/",
    "pose_iter_102000.caffemodel": f"{OPENPOSE_MODELS}/hand/",
}
for fname, dest_dir in mapping.items():
    src = f"{kaggle_path}/{fname}"
    if os.path.exists(src):
        shutil.copy(src, dest_dir)
        print(f"Copiato: {fname} -> {dest_dir}")
    else:
        print(f"MANCANTE: {fname}")


In [ ]:
%cd /content/openpose
!mkdir -p build
%cd build
!cmake .. -DBUILD_PYTHON=ON -DGPU_MODE=CUDA -DDOWNLOAD_BODY_25_MODEL=OFF \
    -DDOWNLOAD_FACE_MODEL=OFF -DDOWNLOAD_HAND_MODEL=OFF -DUSE_CUDNN=OFF

In [ ]:
%cd /content/openpose/build
import multiprocessing
n_cores = multiprocessing.cpu_count()
print(f"Compilazione con {n_cores} core...")
!make -j{n_cores}

In [ ]:
%cd /content/openpose
!rm -rf 3rdparty/pybind11
!git clone -q --depth 1 --branch v2.13.6 https://github.com/pybind/pybind11.git 3rdparty/pybind11

cast_h = "/content/openpose/3rdparty/pybind11/include/pybind11/cast.h"
!sed -i '442s/frame->f_code->co_filename/PyFrame_GetCode(frame)->co_filename/' {cast_h}
!sed -i '444s/frame->f_code->co_name/PyFrame_GetCode(frame)->co_name/' {cast_h}
!sed -i '445s/frame = frame->f_back;/frame = PyFrame_GetBack(frame);/' {cast_h}

pb11_h = "/content/openpose/3rdparty/pybind11/include/pybind11/pybind11.h"
!sed -i '2021s/.*/    PyFrameObject *frame = PyEval_GetFrame();/' {pb11_h}
!sed -i '2022s/frame->f_code->co_name/PyFrame_GetCode(frame)->co_name/' {pb11_h}
!sed -i '2023s/frame->f_code->co_argcount/PyFrame_GetCode(frame)->co_argcount/' {pb11_h}
!sed -i '2026s/.*/            PyFrame_GetLocals(frame), PyTuple_GET_ITEM(PyCode_GetVarnames(PyFrame_GetCode(frame)), 0));/' {pb11_h}

%cd build
!rm -rf python/openpose/CMakeFiles/pyopenpose.dir
!rm -f CMakeCache.txt
!cmake .. -DBUILD_PYTHON=ON -DGPU_MODE=CUDA -DDOWNLOAD_BODY_25_MODEL=OFF \
    -DDOWNLOAD_FACE_MODEL=OFF -DDOWNLOAD_HAND_MODEL=OFF -DUSE_CUDNN=OFF

In [ ]:
%cd /content/openpose/build
!make pyopenpose -j{n_cores}

import shutil, sysconfig, sys

src = "/content/openpose/build/python/openpose/pyopenposeNone"
ext_suffix = sysconfig.get_config_var("EXT_SUFFIX")
dst = f"/content/openpose/build/python/openpose/pyopenpose{ext_suffix}"
shutil.copy(src, dst)
print(f"Copiato in: {dst}")

sys.path.append("/content/openpose/build/python/openpose")
import pyopenpose as op

In [ ]:
params = {
    "model_folder": "/content/openpose/models/",
    "model_pose": "BODY_25",
    "net_resolution": "-1x368",
    "disable_blending": False,
}

opWrapper = op.WrapperPython()
opWrapper.configure(params)
opWrapper.start()

def openpose_pose_fn(frame_bgr):
    """Restituisce (keypoints_array, n_keypoints_rilevati) o (None, 0)"""
    datum = op.Datum()
    datum.cvInputData = frame_bgr
    opWrapper.emplaceAndPop(op.VectorDatum([datum]))

    keypoints = datum.poseKeypoints

    if keypoints is None or len(keypoints) == 0:
        return None, 0

    kp_person = keypoints[0]
    n_detected = int(np.sum(kp_person[:, 2] > 0.5))
    return kp_person, n_detected

summary_openpose, times_openpose, kp_counts_openpose = run_pose_on_video(
    test_video, openpose_pose_fn, max_frames=60
)
print("Test OpenPose (60 frame):")
for k, v in summary_openpose.items():
    print(f"  {k}: {v}")


In [ ]:
%cd /content
!wget -q https://raw.githubusercontent.com/opencv/opencv_extra/4.x/testdata/dnn/openpose_pose_coco.prototxt -O pose_opencv.prototxt
!ls -lh pose_opencv.prototxt

OPENCV_WEIGHTS = f"{kaggle_path}/pose_iter_440000.caffemodel"
print("Pesi disponibili:", os.path.exists(OPENCV_WEIGHTS))


In [ ]:
%cd /content
!git clone -q --depth 1 --branch 4.10.0 https://github.com/opencv/opencv.git opencv_cuda
!git clone -q --depth 1 --branch 4.10.0 https://github.com/opencv/opencv_contrib.git opencv_contrib_cuda

In [ ]:
%cd /content/opencv_cuda
!mkdir -p build
%cd build
!cmake .. \
    -D CMAKE_BUILD_TYPE=RELEASE \
    -D OPENCV_EXTRA_MODULES_PATH=/content/opencv_contrib_cuda/modules \
    -D WITH_CUDA=ON \
    -D WITH_CUDNN=ON \
    -D OPENCV_DNN_CUDA=ON \
    -D ENABLE_FAST_MATH=ON \
    -D CUDA_FAST_MATH=ON \
    -D CUDA_ARCH_BIN=7.5 \
    -D BUILD_opencv_python3=ON \
    -D PYTHON3_EXECUTABLE=$(which python3) \
    -D BUILD_opencv_python2=OFF \
    -D BUILD_TESTS=OFF \
    -D BUILD_PERF_TESTS=OFF \
    -D BUILD_EXAMPLES=OFF \
    -D BUILD_opencv_apps=OFF \
    -D BUILD_DOCS=OFF \
    -D BUILD_opencv_java=OFF \
    -D BUILD_opencv_js=OFF \
    -D BUILD_LIST=core,imgproc,imgcodecs,videoio,dnn,python3,highgui,cudev \
    -D INSTALL_PYTHON_EXAMPLES=OFF

In [ ]:
%cd /content/opencv_cuda/build
!make -j{n_cores}

In [ ]:
CV2_CUDA_PATH = "/content/opencv_cuda/build/lib/python3"
sys.path.insert(0, CV2_CUDA_PATH)

modules_to_remove = [m for m in sys.modules if m.startswith("cv2")]
for m in modules_to_remove:
    del sys.modules[m]

import cv2
print("OpenCV version:", cv2.__version__)
print("OpenCV path:", cv2.__file__)
print("CUDA devices disponibili:", cv2.cuda.getCudaEnabledDeviceCount())

PROTO_PATH = "/content/pose_opencv.prototxt"
WEIGHTS_PATH = OPENCV_WEIGHTS

net_opencv = cv2.dnn.readNetFromCaffe(PROTO_PATH, WEIGHTS_PATH)
net_opencv.setPreferableBackend(cv2.dnn.DNN_BACKEND_CUDA)
net_opencv.setPreferableTarget(cv2.dnn.DNN_TARGET_CUDA)
print("Backend CUDA impostato (build personalizzata)")

N_KEYPOINTS_COCO = 18

def opencv_pose_fn(frame_bgr):
    """Restituisce (keypoints_array, n_keypoints_rilevati) o (None, 0)"""
    frame_h, frame_w = frame_bgr.shape[:2]
    in_height = 368
    in_width = int((in_height / frame_h) * frame_w)

    inp_blob = cv2.dnn.blobFromImage(
        frame_bgr, 1.0 / 255, (in_width, in_height), (0, 0, 0), swapRB=False, crop=False
    )
    net_opencv.setInput(inp_blob)
    output = net_opencv.forward()

    H = output.shape[2]
    W = output.shape[3]

    keypoints = []
    n_detected = 0
    for i in range(N_KEYPOINTS_COCO):
        prob_map = output[0, i, :, :]
        min_val, prob, min_loc, point = cv2.minMaxLoc(prob_map)
        x = (frame_w * point[0]) / W
        y = (frame_h * point[1]) / H

        if prob > 0.1:
            keypoints.append([x, y, prob])
            n_detected += 1
        else:
            keypoints.append([0, 0, 0])

    keypoints = np.array(keypoints)
    if n_detected == 0:
        return None, 0
    return keypoints, n_detected

summary_opencv_cuda, times_opencv_cuda, kp_counts_opencv_cuda = run_pose_on_video(
    test_video, opencv_pose_fn, max_frames=60
)
print("Test OpenCV DNN con CUDA (60 frame):")
for k, v in summary_opencv_cuda.items():
    print(f"  {k}: {v}")


In [ ]:
algorithms = {
    "MediaPipe": mediapipe_pose_fn,
    "YOLO": yolo_pose_fn,
    "OpenPose": openpose_pose_fn,
    "OpenCV_DNN": opencv_pose_fn,
}

all_results = {}

for video_name in video_files:
    video_path = f"{VIDEO_DIR}/{video_name}"
    all_results[video_name] = {}
    print(f"\n=== Video: {video_name} ===")

    for algo_name, algo_fn in algorithms.items():
        print(f"  Eseguo {algo_name}...")
        summary, times, kp_counts = run_pose_on_video(video_path, algo_fn, max_frames=None)
        all_results[video_name][algo_name] = {
            "summary": summary,
            "times": times,
            "kp_counts": kp_counts,
        }
        print(f"    -> {summary['avg_time_per_frame']:.3f}s/frame, "
              f"{summary['avg_keypoints_detected']:.1f} keypoints medi, "
              f"{summary['failed_frames_pct']:.1f}% falliti")

with open(f"{RESULTS_DIR}/all_results_summary.json", "w") as f:
    json_safe = {
        v: {a: r["summary"] for a, r in algos.items()}
        for v, algos in all_results.items()
    }
    json.dump(json_safe, f, indent=2)

print(f"Salvato in {RESULTS_DIR}/all_results_summary.json")


In [ ]:
import pandas as pd

rows = []
for video_name, algos in all_results.items():
    for algo_name, data in algos.items():
        s = data["summary"]
        rows.append({
            "video": video_name,
            "algoritmo": algo_name,
            "tempo_medio_frame_s": round(s["avg_time_per_frame"], 4),
            "fps_elaborazione": round(s["fps_processing"], 2),
            "keypoints_medi": round(s["avg_keypoints_detected"], 1),
            "frame_falliti_pct": round(s["failed_frames_pct"], 2),
            "n_frame_totali": s["n_frames"],
        })

df_results = pd.DataFrame(rows)
df_results.to_csv(f"{RESULTS_DIR}/tabella_riassuntiva.csv", index=False)

print(df_results.to_string(index=False))
print(f"\nSalvato in {RESULTS_DIR}/tabella_riassuntiva.csv")

df_agg = df_results.groupby("algoritmo").agg({
    "tempo_medio_frame_s": "mean",
    "fps_elaborazione": "mean",
    "keypoints_medi": "mean",
    "frame_falliti_pct": "mean",
}).round(3)
df_agg.to_csv(f"{RESULTS_DIR}/tabella_aggregata.csv")

print(df_agg.to_string())


In [ ]:

MEDIAPIPE_CONNECTIONS = mp_pose.POSE_CONNECTIONS

YOLO_CONNECTIONS = [
    (0,1),(0,2),(1,3),(2,4),(5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),(11,13),(13,15),(12,14),(14,16)
]

OPENPOSE_BODY25_CONNECTIONS = [
    (1,8),(1,2),(1,5),(2,3),(3,4),(5,6),(6,7),(8,9),(9,10),(10,11),
    (8,12),(12,13),(13,14),(1,0),(0,15),(15,17),(0,16),(16,18),
    (14,19),(19,20),(14,21),(11,22),(22,23),(11,24)
]

OPENCV_COCO_CONNECTIONS = [
    (1,2),(1,5),(2,3),(3,4),(5,6),(6,7),
    (1,11),(11,12),(12,13),
    (1,8),(8,9),(9,10),
    (1,0),(0,14),(14,16),(0,15),(15,17)
]

def draw_skeleton(frame, keypoints, connections, color, is_mediapipe=False, conf_threshold=0.3):

    img = frame.copy()
    h, w = img.shape[:2]

    points = []
    for kp in keypoints:
        if is_mediapipe:
            x, y = int(kp[0] * w), int(kp[1] * h)
            conf = kp[2]
        else:
            x, y = int(kp[0]), int(kp[1])
            conf = kp[2] if len(kp) > 2 else 1.0

        if conf > conf_threshold and not (x == 0 and y == 0):
            points.append((x, y))
        else:
            points.append(None)

    for conn in connections:
        idx_a, idx_b = conn[0], conn[1]
        if idx_a < len(points) and idx_b < len(points):
            if points[idx_a] and points[idx_b]:
                cv2.line(img, points[idx_a], points[idx_b], color, 2)

    for p in points:
        if p:
            cv2.circle(img, p, 4, color, -1)

    return img

def crop_and_run(video_path, frame_idx, algo_fn, margin=40):
    frames = get_video_frames(video_path, max_frames=frame_idx+1)
    frame = frames[frame_idx]

    results_yolo_box = yolo_model(frame, verbose=False)
    if len(results_yolo_box[0].boxes) == 0:
        return None, None
    box = results_yolo_box[0].boxes.xyxy[0].cpu().numpy()
    x1, y1, x2, y2 = box.astype(int)
    h, w = frame.shape[:2]
    x1 = max(0, x1 - margin); y1 = max(0, y1 - margin)
    x2 = min(w, x2 + margin); y2 = min(h, y2 + margin)
    cropped = frame[y1:y2, x1:x2]

    kp, _ = algo_fn(cropped)
    return cropped, kp


In [ ]:
sample_frames = get_video_frames(test_video, max_frames=1)
sample_frame = sample_frames[0]

kp_mp, _ = mediapipe_pose_fn(sample_frame)
kp_yolo, _ = yolo_pose_fn(sample_frame)
kp_op, _ = openpose_pose_fn(sample_frame)

cropped_frame, kp_cv = crop_and_run(test_video, frame_idx=0, algo_fn=opencv_pose_fn)
if cropped_frame is None:
    cropped_frame, kp_cv = sample_frame, None

img_mp = draw_skeleton(sample_frame, kp_mp, MEDIAPIPE_CONNECTIONS, (0,255,0), is_mediapipe=True)
img_yolo = draw_skeleton(sample_frame, kp_yolo, YOLO_CONNECTIONS, (255,0,0))
img_op = draw_skeleton(sample_frame, kp_op, OPENPOSE_BODY25_CONNECTIONS, (0,0,255))
img_cv = draw_skeleton(cropped_frame, kp_cv, OPENCV_COCO_CONNECTIONS, (255,255,0), conf_threshold=0.2) if kp_cv is not None else cropped_frame

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
titles = ["MediaPipe Pose", "YOLO Pose", "OpenPose", "OpenCV DNN (frame ritagliato)"]
images = [img_mp, img_yolo, img_op, img_cv]

for ax, img, title in zip(axes.flat, images, titles):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=14)
    ax.axis("off")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/confronto_skeleton.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Salvato in {RESULTS_DIR}/confronto_skeleton.png")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 8))

for ax, video_name in zip(axes, video_files):
    video_path = f"{VIDEO_DIR}/{video_name}"
    cropped, kp = crop_and_run(video_path, frame_idx=30, algo_fn=opencv_pose_fn)
    if cropped is not None and kp is not None:
        img = draw_skeleton(cropped, kp, OPENCV_COCO_CONNECTIONS, (255,255,0), conf_threshold=0.2)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"OpenCV DNN - {video_name}", fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/opencv_dnn_multi_video.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Salvato in {RESULTS_DIR}/opencv_dnn_multi_video.png")


In [ ]:
def find_hardest_frame(video_path, sample_every=10):
    frames = get_video_frames(video_path)
    worst_idx, worst_score = 0, 999
    for i in range(0, len(frames), sample_every):
        kp, n_detected = mediapipe_pose_fn(frames[i])
        if n_detected < worst_score:
            worst_score = n_detected
            worst_idx = i
    return worst_idx, frames[worst_idx], worst_score

hard_frames = {}
for video_name in video_files:
    video_path = f"{VIDEO_DIR}/{video_name}"
    idx, frame, score = find_hardest_frame(video_path)
    hard_frames[video_name] = (idx, frame)
    print(f"{video_name}: frame piu' difficile = #{idx} ({score} keypoint MediaPipe rilevati)")


In [ ]:
def compare_all_algos_on_frame(frame, title_prefix=""):
    kp_mp, n_mp = mediapipe_pose_fn(frame)
    kp_yolo, n_yolo = yolo_pose_fn(frame)
    kp_op, n_op = openpose_pose_fn(frame)

    results_yolo_box = yolo_model(frame, verbose=False)
    if len(results_yolo_box[0].boxes) > 0:
        box = results_yolo_box[0].boxes.xyxy[0].cpu().numpy()
        x1, y1, x2, y2 = box.astype(int)
        h, w = frame.shape[:2]
        margin = 40
        x1 = max(0, x1-margin); y1 = max(0, y1-margin)
        x2 = min(w, x2+margin); y2 = min(h, y2+margin)
        cropped = frame[y1:y2, x1:x2]
        kp_cv, n_cv = opencv_pose_fn(cropped)
    else:
        cropped, kp_cv, n_cv = frame, None, 0

    img_mp = draw_skeleton(frame, kp_mp, MEDIAPIPE_CONNECTIONS, (0,255,0), is_mediapipe=True)
    img_yolo = draw_skeleton(frame, kp_yolo, YOLO_CONNECTIONS, (255,0,0))
    img_op = draw_skeleton(frame, kp_op, OPENPOSE_BODY25_CONNECTIONS, (0,0,255))
    img_cv = draw_skeleton(cropped, kp_cv, OPENCV_COCO_CONNECTIONS, (255,255,0), conf_threshold=0.2) if kp_cv is not None else cropped

    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    titles = [f"MediaPipe ({n_mp} kp)", f"YOLO ({n_yolo} kp)", f"OpenPose ({n_op} kp)", f"OpenCV DNN ({n_cv} kp)"]
    images = [img_mp, img_yolo, img_op, img_cv]
    for ax, img, t in zip(axes.flat, images, titles):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(t, fontsize=12)
        ax.axis("off")
    fig.suptitle(title_prefix, fontsize=14)
    plt.tight_layout()
    return fig

for video_name, (idx, frame) in hard_frames.items():
    fig = compare_all_algos_on_frame(frame, title_prefix=f"{video_name} - frame #{idx} (condizioni difficili)")
    safe_name = video_name.replace(" ", "_").replace(".mp4", "")
    fig.savefig(f"{RESULTS_DIR}/robustezza_{safe_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
from google.colab import files as colab_files
import shutil

shutil.make_archive("/content/results_finale", "zip", RESULTS_DIR)
print("Archivio creato: /content/results_finale.zip")
colab_files.download("/content/results_finale.zip")
